# Обзор данных

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

GAP_DAYS = 14          # лаг между последним допустимым заказом и scoring_date
HISTORY_YEARS = 2      # глубина допустимой истории
TARGET_MONTHS = 3      # длина целевого окна
ACTIVE_DAYS = 365      # правило отбора: есть заказ за последние 365 дней до отсечения


def average_precision_at_20_percent(customer_id, target, score):
    ranking = (
        pd.DataFrame({"customer_id": np.asarray(customer_id).astype(str), "target": np.asarray(target), "score": np.asarray(score)})
        .sort_values(["score", "customer_id"], ascending=[False, True], kind="mergesort")
        .reset_index(drop=True)
    )
    k = int(np.ceil(len(ranking) * 0.20))
    top_target = ranking["target"].iloc[:k].to_numpy()
    precision_at_rank = np.cumsum(top_target) / np.arange(1, k + 1)
    denominator = min(k, int(ranking["target"].sum()))
    return (precision_at_rank * top_target).sum() / max(denominator, 1)

## Загрузка и общий вид таблиц

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

tables = {"train": train, "test": test, "sample_submission": sample, "orders": orders, "products": products}


def overview(df):
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing": df.isna().sum(),
            "missing_%": df.isna().mean() * 100,
            "nunique": df.nunique(),
        }
    )


for name, df in tables.items():
    mem = df.memory_usage(deep=True).sum() / 2**20
    print(f"\n=== {name}: {df.shape[0]:,} строк x {df.shape[1]} колонок, {mem:.1f} МБ")
    display(overview(df))
    display(df.head(3))

## Ключи, дубли и связи между таблицами

In [ ]:
checks = {}

checks["train: дубли (customer_id, scoring_date)"] = int(train.duplicated(["customer_id", "scoring_date"]).sum())
checks["test: дубли (customer_id, scoring_date)"] = int(test.duplicated(["customer_id", "scoring_date"]).sum())
checks["test: дубли customer_id"] = int(test["customer_id"].duplicated().sum())
checks["sample: набор ID совпадает с test"] = set(sample["customer_id"]) == set(test["customer_id"])
checks["sample: порядок ID совпадает с test"] = bool(
    len(sample) == len(test) and (sample["customer_id"].to_numpy() == test["customer_id"].to_numpy()).all()
)

checks["products: дубли product_id"] = int(products["product_id"].duplicated().sum())
checks["orders: полные дубли строк"] = int(orders.duplicated().sum())
checks["orders: дубли (order_id, product_id)"] = int(orders.duplicated(["order_id", "product_id"]).sum())

per_order = orders.groupby("order_id").agg(
    n_customers=("customer_id", "nunique"),
    n_dates=("order_date", "nunique"),
    n_lines=("product_id", "size"),
)
checks["orders: заказов с >1 клиентом"] = int((per_order["n_customers"] > 1).sum())
checks["orders: заказов с >1 датой"] = int((per_order["n_dates"] > 1).sum())

known_products = set(products["product_id"])
checks["orders: доля позиций без товара в справочнике, %"] = round(
    float((~orders["product_id"].isin(known_products)).mean() * 100), 4
)
checks["products: товаров, которых нет в заказах"] = int((~products["product_id"].isin(orders["product_id"])).sum())

slice_customers = set(train["customer_id"]) | set(test["customer_id"])
order_customers = set(orders["customer_id"])
checks["клиентов в orders всего"] = len(order_customers)
checks["клиентов в orders, которых нет в train/test"] = len(order_customers - slice_customers)
checks["клиентов train/test без единой строки в orders"] = len(slice_customers - order_customers)

display(pd.Series(checks, name="значение").to_frame())

print("Позиций в заказе:")
display(per_order["n_lines"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T)

## Значения колонок

In [ ]:
num_cols = ["paid_price", "loyalty_percent", "discount_percent"]
display(orders[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

display(
    pd.DataFrame(
        {
            "< 0": (orders[num_cols] < 0).sum(),
            "== 0": (orders[num_cols] == 0).sum(),
            "== 0, %": (orders[num_cols] == 0).mean() * 100,
            "> 100": (orders[num_cols] > 100).sum(),
        }
    )
)

print("Самые частые значения loyalty_percent:", orders["loyalty_percent"].value_counts(normalize=True).head(10).round(4).to_dict())
print("Самые частые значения discount_percent:", orders["discount_percent"].value_counts(normalize=True).head(10).round(4).to_dict())
print("Клиентов с более чем одним значением loyalty_percent, %:",
      round(float((orders.groupby("customer_id")["loyalty_percent"].nunique() > 1).mean() * 100), 2))
print("Заказов с более чем одним значением loyalty_percent, %:",
      round(float((orders.groupby("order_id")["loyalty_percent"].nunique() > 1).mean() * 100), 2))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(np.log1p(orders["paid_price"].clip(lower=0)), bins=80)
axes[0].set_title("log1p(paid_price)")
axes[1].hist(orders["loyalty_percent"].dropna(), bins=50)
axes[1].set_title("loyalty_percent")
axes[2].hist(orders["discount_percent"].dropna(), bins=50)
axes[2].set_title("discount_percent")
plt.tight_layout()
plt.show()

In [ ]:
for col in [c for c in products.columns if c != "product_id"]:
    vc = products[col].value_counts(dropna=False)
    print(f"\n--- products.{col}: {products[col].nunique()} уникальных, пропусков {products[col].isna().sum()}")
    display(vc.head(12).to_frame("товаров").T)

## Период данных и сезонность

In [ ]:
print("order_date:", orders["order_date"].min().date(), "…", orders["order_date"].max().date())
print("scoring_date в train:", sorted(train["scoring_date"].dt.date.unique()))
print("scoring_date в test: ", sorted(test["scoring_date"].dt.date.unique()))

monthly = (
    orders.assign(month=orders["order_date"].dt.to_period("M"))
    .groupby("month")
    .agg(
        lines=("product_id", "size"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        paid_sum=("paid_price", "sum"),
    )
)
monthly["avg_order_value"] = monthly["paid_sum"] / monthly["orders"]
display(monthly)

ax = monthly[["orders", "customers"]].plot(figsize=(14, 4), marker="o", title="Заказы и активные клиенты по месяцам")
plt.show()

dow = orders.drop_duplicates("order_id")["order_date"].dt.dayofweek.value_counts(normalize=True).sort_index()
print("Доля заказов по дням недели (0 = понедельник):", dow.round(3).to_dict())

## Срезы и пересечение клиентов

In [ ]:
slices = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
slices["slice"] = slices["scoring_date"].dt.strftime("%Y-%m-%d")

display(
    slices.groupby(["part", "slice"]).agg(
        rows=("customer_id", "size"),
        customers=("customer_id", "nunique"),
        target_rate=("target", "mean"),
    )
)

ids = {s: set(g["customer_id"]) for s, g in slices.groupby("slice")}
names = sorted(ids)
inter = pd.DataFrame([[len(ids[a] & ids[b]) for b in names] for a in names], index=names, columns=names)
print("Пересечение клиентов между срезами (на диагонали — размер среза):")
display(inter)

n_slices = slices.groupby("customer_id")["slice"].nunique()
print("Сколько клиентов встречается в 1, 2, 3, 4 срезах:", n_slices.value_counts().sort_index().to_dict())

## Таблица заказов и зоны относительно даты скоринга

In [ ]:
order_tbl = (
    orders.groupby(["customer_id", "order_id"], as_index=False)
    .agg(
        order_date=("order_date", "min"),
        order_sum=("paid_price", "sum"),
        n_items=("product_id", "size"),
        n_products=("product_id", "nunique"),
        discount_mean=("discount_percent", "mean"),
        discount_max=("discount_percent", "max"),
        loyalty_mean=("loyalty_percent", "mean"),
    )
    .sort_values(["customer_id", "order_date", "order_id"])
    .reset_index(drop=True)
)
print(f"Заказов: {len(order_tbl):,}, клиентов: {order_tbl['customer_id'].nunique():,}")
display(order_tbl[["order_sum", "n_items", "discount_mean", "loyalty_mean"]].describe(percentiles=[0.05, 0.5, 0.95, 0.99]).T)

per_customer = order_tbl.groupby("customer_id").agg(n_orders=("order_id", "size"), n_days=("order_date", "nunique"))
print("Заказов на клиента за всё время:")
display(per_customer["n_orders"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).to_frame().T)
print("Доля клиентов, у которых бывает несколько заказов в один день, %:",
      round(float((per_customer["n_orders"] > per_customer["n_days"]).mean() * 100), 2))

In [ ]:
# Зоны: 0_too_old — раньше (cutoff − 2 года); 1_history — до cutoff включительно (можно использовать);
# 2_gap — 14 дней перед scoring_date; 3_target — (scoring_date, scoring_date + 3 месяца]; 4_after — позже.
bounds = pd.DataFrame({"scoring_date": sorted(slices["scoring_date"].unique())})
bounds["cutoff"] = bounds["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
bounds["history_start"] = bounds["cutoff"].map(lambda d: d - pd.DateOffset(years=HISTORY_YEARS))
bounds["target_end"] = bounds["scoring_date"].map(lambda d: d + pd.DateOffset(months=TARGET_MONTHS))
display(bounds)

slices = slices.merge(bounds, on="scoring_date", how="left")
pairs = slices.merge(order_tbl[["customer_id", "order_id", "order_date", "order_sum"]], on="customer_id", how="left")
od = pairs["order_date"]
pairs["zone"] = np.select(
    [
        od.isna(),
        od < pairs["history_start"],
        od <= pairs["cutoff"],
        od <= pairs["scoring_date"],
        od <= pairs["target_end"],
    ],
    ["no_orders", "0_too_old", "1_history", "2_gap", "3_target"],
    default="4_after",
)
pairs["days_before_cutoff"] = (pairs["cutoff"] - pairs["order_date"]).dt.days

print("Число заказов по зонам:")
display(pairs.pivot_table(index="slice", columns="zone", values="order_id", aggfunc="count", fill_value=0))
print("Первая и последняя дата заказа у клиентов каждого среза:")
display(pairs.groupby("slice")["order_date"].agg(["min", "max"]))
print("Заказов позже даты отсечения своего клиента:", int((pairs["days_before_cutoff"] < 0).sum()))

## Первые сигналы: давность и частота

In [ ]:
hist = pairs[pairs["zone"].eq("1_history")]
rf = hist.groupby(["slice", "customer_id"]).agg(
    recency=("days_before_cutoff", "min"),
    tenure=("days_before_cutoff", "max"),
    n_orders=("order_id", "nunique"),
    n_orders_90d=("days_before_cutoff", lambda s: int((s <= 90).sum())),
    n_orders_365d=("days_before_cutoff", lambda s: int((s <= 365).sum())),
)
quick = slices.merge(rf.reset_index(), on=["slice", "customer_id"], how="left")
quick["has_history"] = quick["recency"].notna()

print("Доля строк без единого заказа в допустимой истории, %:")
display((1 - quick.groupby(["part", "slice"])["has_history"].mean()).mul(100).to_frame("no_history_%"))
display(quick.groupby(["part", "slice"])[["recency", "tenure", "n_orders", "n_orders_90d"]].describe(percentiles=[0.25, 0.5, 0.75]).T)

print("Доля клиентов по времени с первого заказа (tenure), %:")
display(
    quick.assign(tenure_bin=pd.cut(quick["tenure"], [-1, 30, 90, 180, 365, 10_000], labels=["0-30", "31-90", "91-180", "181-365", "365+"]))
    .pivot_table(index="tenure_bin", columns="slice", values="customer_id", aggfunc="count", observed=False)
    .pipe(lambda t: t / t.sum() * 100)
)

In [ ]:
REC_EDGES = [-1, 7, 14, 30, 60, 90, 180, 365, 10_000]
REC_LABELS = ["0-7", "8-14", "15-30", "31-60", "61-90", "91-180", "181-365", "365+"]
N_EDGES = [-1, 0, 1, 2, 3, 5, 9, 10_000]
N_LABELS = ["0", "1", "2", "3", "4-5", "6-9", "10+"]

qt = quick[quick["part"].eq("train")].copy()
qt["recency_bin"] = pd.cut(qt["recency"], REC_EDGES, labels=REC_LABELS)
qt["n_orders_bin"] = pd.cut(qt["n_orders"].fillna(0), N_EDGES, labels=N_LABELS)

by_rec = qt.groupby("recency_bin", observed=True)["target"].agg(["mean", "size"])
by_freq = qt.groupby("n_orders_bin", observed=True)["target"].agg(["mean", "size"])
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
by_rec["mean"].plot.bar(ax=axes[0], title="Доля target = 1 по давности последнего заказа, дни")
by_freq["mean"].plot.bar(ax=axes[1], title="Доля target = 1 по числу заказов в истории")
plt.tight_layout()
plt.show()
display(by_rec.T)
display(by_freq.T)

print("Доля target = 1: давность x число заказов")
display(qt.pivot_table(index="recency_bin", columns="n_orders_bin", values="target", aggfunc="mean", observed=True))
print("Число клиентов: давность x число заказов")
display(qt.pivot_table(index="recency_bin", columns="n_orders_bin", values="target", aggfunc="size", observed=True))

rows = []
for s, g in qt.groupby("slice"):
    rec = g["recency"].fillna(10_000)
    rows.append(
        {
            "slice": s,
            "target_rate": g["target"].mean(),
            "случайно": average_precision_at_20_percent(g["customer_id"], g["target"], np.random.default_rng(0).random(len(g))),
            "-recency": average_precision_at_20_percent(g["customer_id"], g["target"], -rec),
            "n_orders": average_precision_at_20_percent(g["customer_id"], g["target"], g["n_orders"].fillna(0)),
            "n_orders_90d": average_precision_at_20_percent(g["customer_id"], g["target"], g["n_orders_90d"].fillna(0)),
            "n_orders / (recency + 30)": average_precision_at_20_percent(
                g["customer_id"], g["target"], g["n_orders"].fillna(0) / (rec + 30)
            ),
        }
    )
print("AP@20% одиночных признаков по срезам train:")
display(pd.DataFrame(rows).set_index("slice"))

## Структура идентификаторов

In [ ]:
print("customer_id:", train["customer_id"].min(), "…", max(train["customer_id"].max(), test["customer_id"].max()))
display(slices.groupby("slice")["customer_id"].describe(percentiles=[0.5]))

oid = np.sort(order_tbl["order_id"].to_numpy())
print(f"order_id: {oid[0]} … {oid[-1]}, уникальных {len(oid):,}, пропущенных номеров {oid[-1] - oid[0] + 1 - len(oid):,}")
pid = np.sort(products["product_id"].to_numpy())
print(f"product_id: {pid[0]} … {pid[-1]}, уникальных {len(pid):,}, пропущенных номеров {pid[-1] - pid[0] + 1 - len(pid):,}")

ot = order_tbl.merge(slices[["customer_id", "slice", "part", "cutoff"]], on="customer_id", how="left")
ot["date_num"] = (ot["order_date"] - ot["order_date"].min()).dt.days
ot["days_before_cutoff"] = (ot["cutoff"] - ot["order_date"]).dt.days

print("order_id по срезам:")
display(ot.groupby("slice")["order_id"].describe(percentiles=[0.5]))
print("Корреляция Спирмена:")
display(ot[["order_id", "customer_id", "date_num", "days_before_cutoff", "order_sum", "n_items"]].corr(method="spearman"))

print("Число заказов: дециль order_id x квартал")
display(pd.crosstab(pd.qcut(ot["order_id"], 10, labels=False), ot["order_date"].dt.to_period("Q")))
print("Число заказов: дециль order_id x срез")
display(pd.crosstab(pd.qcut(ot["order_id"], 10, labels=False), ot["slice"]))

nxt = ot.sort_values(["customer_id", "order_date", "order_id"])
same_c = nxt["customer_id"].eq(nxt["customer_id"].shift())
later = nxt["order_date"].gt(nxt["order_date"].shift())
id_up = nxt["order_id"].gt(nxt["order_id"].shift())
print("Доля пар соседних по времени заказов клиента, где order_id растёт:", round(float(id_up[same_c & later].mean()), 4))
id_step = (nxt["order_id"] - nxt["order_id"].shift())[same_c & later].abs()
print("Модуль разницы order_id у соседних заказов клиента:")
display(id_step.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame().T)

sample_ids = ot.sample(min(len(ot), 40_000), random_state=0)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].scatter(sample_ids["order_date"], sample_ids["order_id"], s=1, alpha=0.3)
axes[0].set_title("order_id от даты заказа")
axes[1].scatter(sample_ids["customer_id"], sample_ids["order_id"], s=1, alpha=0.3)
axes[1].set_title("order_id от customer_id")
plt.tight_layout()
plt.show()

In [ ]:
# Пропуски в нумерации заказов: сколько номеров отсутствует сразу после и сразу перед каждым order_id
gaps = pd.DataFrame(
    {
        "order_id": oid,
        "gap_after": np.diff(oid, append=oid[-1] + 1) - 1,
        "gap_before": np.diff(oid, prepend=oid[0] - 1) - 1,
    }
)
print("Распределение gap_after:", gaps["gap_after"].value_counts().sort_index().head(15).to_dict())
ot = ot.merge(gaps, on="order_id", how="left")

# Порядок строк в исходных файлах
orders["row"] = np.arange(len(orders))
row_stats = orders.groupby("order_id")["row"].agg(["min", "max", "size"])
print("Доля заказов, чьи позиции идут в файле подряд, %:",
      round(float(((row_stats["max"] - row_stats["min"] + 1) == row_stats["size"]).mean() * 100), 2))
print("Корреляция Спирмена номера строки orders с колонками:")
display(
    orders.assign(date_num=(orders["order_date"] - orders["order_date"].min()).dt.days)[
        ["row", "customer_id", "order_id", "product_id", "date_num", "paid_price"]
    ].corr(method="spearman").loc[["row"]]
)
print("train отсортирован по (scoring_date, customer_id):",
      bool(train.equals(train.sort_values(["scoring_date", "customer_id"], kind="mergesort"))))
print("test отсортирован по customer_id:", bool(test["customer_id"].is_monotonic_increasing))

In [ ]:
# AP@20% «неповеденческих» признаков: всё, что заметно выше случайного уровня и не объясняется частотой и давностью, подозрительно
last_order = ot.sort_values(["customer_id", "order_date", "order_id"]).groupby("customer_id").tail(1).set_index("customer_id")
g_ot = ot.groupby("customer_id")
g_rows = orders.groupby("customer_id")
idf = pd.DataFrame(
    {
        "order_id_min": g_ot["order_id"].min(),
        "order_id_max": g_ot["order_id"].max(),
        "order_id_mean": g_ot["order_id"].mean(),
        "order_id_last": last_order["order_id"],
        "gap_after_sum": g_ot["gap_after"].sum(),
        "gap_after_max": g_ot["gap_after"].max(),
        "gap_after_last": last_order["gap_after"],
        "gap_before_sum": g_ot["gap_before"].sum(),
        "gap_before_last": last_order["gap_before"],
        "product_id_min": g_rows["product_id"].min(),
        "product_id_max": g_rows["product_id"].max(),
        "product_id_mean": g_rows["product_id"].mean(),
        "orders_row_min": g_rows["row"].min(),
        "orders_row_max": g_rows["row"].max(),
        "orders_row_mean": g_rows["row"].mean(),
    }
).reset_index()

idq = (
    train.assign(train_row=np.arange(len(train)))
    .merge(quick[["customer_id", "slice", "recency", "n_orders"]], on="customer_id", how="left")
    .merge(idf, on="customer_id", how="left")
)
idq["customer_id_value"] = idq["customer_id"]
idq["behaviour: n_orders / (recency + 30)"] = idq["n_orders"] / (idq["recency"] + 30)

id_cols = ["behaviour: n_orders / (recency + 30)", "customer_id_value", "train_row"] + [c for c in idf.columns if c != "customer_id"]
rows = []
for col in id_cols:
    row = {"feature": col}
    for s, g in idq.groupby("slice"):
        up = average_precision_at_20_percent(g["customer_id"], g["target"], g[col])
        down = average_precision_at_20_percent(g["customer_id"], g["target"], -g[col])
        row[f"{s} ↑"] = up
        row[f"{s} ↓"] = down
    row["corr с n_orders"] = idq[col].corr(idq["n_orders"], method="spearman")
    row["corr с recency"] = idq[col].corr(idq["recency"], method="spearman")
    rows.append(row)
print("AP@20% при сортировке по возрастанию (↑) и убыванию (↓) признака:")
display(pd.DataFrame(rows).set_index("feature"))

print("Доля target = 1 по децилям customer_id:")
display(idq.assign(decile=pd.qcut(idq["customer_id"], 10, labels=False)).pivot_table(index="decile", columns="slice", values="target"))
print("Доля target = 1 по gap_after последнего заказа клиента:")
display(
    idq.assign(gap=idq["gap_after_last"].clip(upper=5)).groupby(["slice", "gap"])["target"].agg(["mean", "size"]).unstack(0)
)

## Активность по срезам

In [ ]:
hist = pairs[pairs["zone"].eq("1_history")].copy()
hist["month"] = hist["order_date"].dt.to_period("M")
hist["months_before_cutoff"] = hist["days_before_cutoff"] // 30
hist["weeks_before_cutoff"] = hist["days_before_cutoff"] // 7

print("Заказы по календарным месяцам и срезам (последние 9 месяцев):")
display(hist.pivot_table(index="month", columns="slice", values="order_id", aggfunc="count", fill_value=0).tail(9))

print("Заказы по 30-дневным интервалам до отсечения (0 = последние 30 дней):")
display(hist.pivot_table(index="months_before_cutoff", columns="slice", values="order_id", aggfunc="count", fill_value=0).head(14))
print("Активные клиенты по 30-дневным интервалам до отсечения:")
display(hist.pivot_table(index="months_before_cutoff", columns="slice", values="customer_id", aggfunc="nunique", fill_value=0).head(14))
print("Средний чек по 30-дневным интервалам до отсечения:")
display(hist.pivot_table(index="months_before_cutoff", columns="slice", values="order_sum", aggfunc="mean").head(14))

print("Заказы по неделям до отсечения (0 = последняя неделя):")
display(hist.pivot_table(index="weeks_before_cutoff", columns="slice", values="order_id", aggfunc="count", fill_value=0).head(10))
print("Средний чек по неделям до отсечения:")
display(hist.pivot_table(index="weeks_before_cutoff", columns="slice", values="order_sum", aggfunc="mean").head(10))

print("Доля клиентов по давности последнего заказа, %:")
display(
    quick.assign(recency_bin=pd.cut(quick["recency"], REC_EDGES, labels=REC_LABELS))
    .pivot_table(index="recency_bin", columns="slice", values="customer_id", aggfunc="count", observed=True)
    .pipe(lambda t: t / t.sum() * 100)
)
print("Доля клиентов по числу заказов в истории, %:")
display(
    quick.assign(n_bin=pd.cut(quick["n_orders"].fillna(0), N_EDGES, labels=N_LABELS))
    .pivot_table(index="n_bin", columns="slice", values="customer_id", aggfunc="count", observed=True)
    .pipe(lambda t: t / t.sum() * 100)
)

## Сдвинутые срезы

In [ ]:
# Сдвигаем дату скоринга назад так, чтобы целевое окно целиком лежало в известной истории клиента.
# shift = 0: целевое окно заканчивается на дате отсечения клиента; shift = k: ещё на k месяцев раньше.
# unbiased — клиенты, которые попали бы в выборку независимо от покупок после сдвинутой даты:
# их последний заказ на сдвинутую дату уже лежит в окне 365 дней до настоящего отсечения.
def stats_at(base):
    m = base[["customer_id", "p_cutoff", "p_date", "p_end"]].merge(order_tbl[["customer_id", "order_date"]], on="customer_id")
    h = m[m["order_date"] <= m["p_cutoff"]]
    out = base.set_index("customer_id").copy()
    out["last_hist"] = h.groupby("customer_id")["order_date"].max()
    out["n_all"] = h.groupby("customer_id").size()
    out["n_365"] = h[h["order_date"] >= h["p_cutoff"] - pd.Timedelta(days=ACTIVE_DAYS)].groupby("customer_id").size()
    t = m[(m["order_date"] > m["p_date"]) & (m["order_date"] <= m["p_end"])]
    out["target_orders"] = t.groupby("customer_id").size()
    out["target_from_orders"] = (out["target_orders"].fillna(0) > 0).astype(int)
    out[["n_all", "n_365"]] = out[["n_all", "n_365"]].fillna(0)
    out["recency"] = (out["p_cutoff"] - out["last_hist"]).dt.days
    out["eligible"] = out["recency"].le(ACTIVE_DAYS)
    out["unbiased"] = out["eligible"] & (out["last_hist"] >= out["cutoff"] - pd.Timedelta(days=ACTIVE_DAYS))
    out["recency_bin"] = pd.cut(out["recency"], REC_EDGES, labels=REC_LABELS)
    out["n_365_bin"] = pd.cut(out["n_365"], N_EDGES, labels=N_LABELS)
    return out.reset_index()


base_cols = ["customer_id", "part", "slice", "target", "cutoff"]
real = slices[base_cols].assign(p_cutoff=slices["cutoff"], p_date=slices["scoring_date"], p_end=slices["target_end"])
real = stats_at(real)
real["y"] = real["target"]

SHIFTS = [0, 1, 2, 3]
pseudo = {}
for k in SHIFTS:
    b = slices[base_cols].copy()
    b["p_end"] = b["cutoff"] - pd.DateOffset(months=k)
    b["p_date"] = b["p_end"] - pd.DateOffset(months=TARGET_MONTHS)
    b["p_cutoff"] = b["p_date"] - pd.Timedelta(days=GAP_DAYS)
    pseudo[k] = stats_at(b)
    pseudo[k]["y"] = pseudo[k]["target_from_orders"]

rows = []
for k, p in pseudo.items():
    for part, g in p.groupby("part"):
        u = g[g["unbiased"]]
        rows.append(
            {
                "shift": k,
                "part": part,
                "сдвинутые даты скоринга": ", ".join(sorted(g["p_date"].dt.strftime("%Y-%m-%d").unique())),
                "клиентов всего": len(g),
                "eligible": int(g["eligible"].sum()),
                "unbiased": len(u),
                "макс. давность unbiased": u["recency"].max(),
                "доля y=1 eligible": g.loc[g["eligible"], "y"].mean(),
                "доля y=1 unbiased": u["y"].mean(),
                "AP@20% формулы на unbiased": average_precision_at_20_percent(u["customer_id"], u["y"], u["n_all"] / (u["recency"] + 30)),
            }
        )
display(pd.DataFrame(rows).set_index(["shift", "part"]))

In [ ]:
real_train = real[real["part"].eq("train")]
cmp_rec = {"настоящий train": real_train.groupby("recency_bin", observed=True)["y"].mean()}
cmp_rec_n = {"настоящий train": real_train.groupby("recency_bin", observed=True)["y"].size()}
for k, p in pseudo.items():
    u = p[p["unbiased"]]
    cmp_rec[f"shift={k} unbiased"] = u.groupby("recency_bin", observed=True)["y"].mean()
    cmp_rec_n[f"shift={k} unbiased"] = u.groupby("recency_bin", observed=True)["y"].size()
cmp_rec["shift=0 eligible (со смещением отбора)"] = pseudo[0][pseudo[0]["eligible"]].groupby("recency_bin", observed=True)["y"].mean()
print("Доля y = 1 по давности: настоящий train и сдвинутые срезы")
display(pd.DataFrame(cmp_rec))
print("Число клиентов в ячейках:")
display(pd.DataFrame(cmp_rec_n))

REC_LIMIT = 150  # в этом диапазоне давности unbiased-часть полна для всех сдвигов
cmp_n = {"настоящий train": real_train[real_train["recency"] <= REC_LIMIT].groupby("n_365_bin", observed=True)["y"].mean()}
for k, p in pseudo.items():
    u = p[p["unbiased"] & (p["recency"] <= REC_LIMIT)]
    cmp_n[f"shift={k} unbiased"] = u.groupby("n_365_bin", observed=True)["y"].mean()
print(f"Доля y = 1 по числу заказов за 365 дней (только давность <= {REC_LIMIT}):")
display(pd.DataFrame(cmp_n))

print("Доля y = 1 в сдвинутых срезах отдельно по исходным срезам (unbiased, давность <= 150):")
display(
    pd.concat([p[p["unbiased"] & (p["recency"] <= REC_LIMIT)].assign(shift=k) for k, p in pseudo.items()])
    .pivot_table(index="shift", columns="slice", values="y", aggfunc="mean")
)
print("Для сравнения, настоящий train при давности <= 150:")
display(real_train[real_train["recency"] <= REC_LIMIT].groupby("slice")["y"].mean().to_frame().T)

## Предобработка и сохранение

In [ ]:
attr_cols = [c for c in products.columns if c != "product_id"]
products_clean = products.drop_duplicates("product_id").copy()
for c in attr_cols:
    products_clean[c] = products_clean[c].astype("object").where(products_clean[c].notna(), "unknown").astype(str)

order_lines = orders.drop(columns="row").merge(products_clean, on="product_id", how="left", validate="m:1")
assert len(order_lines) == len(orders), "merge изменил число строк"
for c in attr_cols:
    order_lines[c] = order_lines[c].fillna("unknown").astype("category")
for c in ["paid_price", "loyalty_percent", "discount_percent"]:
    order_lines[c] = order_lines[c].astype("float32")
order_lines = order_lines.sort_values(["customer_id", "order_date", "order_id"]).reset_index(drop=True)

order_lines.to_parquet(OUT_DIR / "order_lines.parquet", index=False)
order_tbl.to_parquet(OUT_DIR / "order_tbl.parquet", index=False)

for name, df in {"order_lines": order_lines, "order_tbl": order_tbl}.items():
    print(f"{name}: {df.shape}, {df.memory_usage(deep=True).sum() / 2**20:.1f} МБ")
display(order_lines.head())
display(order_tbl.head())

## Ряды покупок

In [ ]:
# Гипотеза: правило «последняя покупка + средний интервал попадает в окно» действует не на клиента целиком,
# а на отдельные ряды покупок (по категории, бренду, товару). Ячейке нужны только train, test, orders, products.
ZONE_LO, ZONE_HI = 25, 92   # зона, найденная для клиентов с 2–3 заказами: дни после отсечения
SER_N_EDGES = [0, 1, 2, 3, 5, 9, 20, 10_000]
SER_N_LABELS = ["1", "2", "3", "4-5", "6-9", "10-20", "21+"]

sc = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
sc["cutoff"] = sc["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
L = orders[["customer_id", "order_date", "product_id"]].merge(products, on="product_id", how="left").merge(
    sc[["customer_id", "cutoff"]], on="customer_id", how="inner"
)
L["d"] = (L["cutoff"] - L["order_date"]).dt.days
L["all"] = "all"
L["gender_category"] = L["gender_product"].astype(str) + "|" + L["category"].astype(str)
L["gender_category2"] = L["gender_product"].astype(str) + "|" + L["category2"].astype(str)

cs = sc[sc["part"].eq("train")][["customer_id", "target"]].copy()
cs["n_days"] = cs["customer_id"].map(L.drop_duplicates(["customer_id", "d"]).groupby("customer_id").size())
cs["n_bin"] = pd.cut(cs["n_days"], SER_N_EDGES, labels=SER_N_LABELS)

LEVELS = {
    "клиент целиком": "all", "пол товара": "gender_product", "категория": "category", "пол x категория": "gender_category",
    "подкатегория": "category2", "пол x подкатегория": "gender_category2", "бренд": "brand", "товар": "product_id",
}
series = {}
for title, key in LEVELS.items():
    st = L[["customer_id", key, "d"]].drop_duplicates().groupby(["customer_id", key], observed=True)["d"].agg(n="size", last="min", first="max")
    st = st[st["n"] >= 2].reset_index()
    st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
    st["t"] = st["gap"] - st["last"]   # ожидаемый день следующей покупки в ряду, от отсечения
    st["in_zone"] = st["t"].gt(ZONE_LO) & st["t"].le(ZONE_HI)
    st["sharp"] = st["in_zone"] & st["n"].le(3)
    series[title] = st
    cs[f"sharp|{title}"] = cs["customer_id"].map(st.groupby("customer_id")["sharp"].sum()).fillna(0)
    cs[f"zone|{title}"] = cs["customer_id"].map(st.groupby("customer_id")["in_zone"].sum()).fillna(0)

for kind, text in [("sharp", "ряд из 2–3 покупок, ожидаемый день в зоне"), ("zone", "ряд любой длины, ожидаемый день в зоне")]:
    hit, miss, share = {}, {}, {}
    for title in LEVELS:
        has = cs[f"{kind}|{title}"] > 0
        hit[title] = cs[has].groupby("n_bin", observed=True)["target"].mean()
        miss[title] = cs[~has].groupby("n_bin", observed=True)["target"].mean()
        share[title] = has.groupby(cs["n_bin"], observed=True).mean()
    print(f"=== {text}")
    print("Доля target = 1 у клиентов, у которых есть хотя бы один такой ряд (столбцы — число дней с заказами):")
    display(pd.DataFrame(hit).T)
    print("Доля target = 1 у остальных:")
    display(pd.DataFrame(miss).T)
    print("Доля клиентов с таким рядом:")
    display(pd.DataFrame(share).T)

for title in ["подкатегория", "бренд"]:
    print(f"Доля target = 1 по числу рядов из 2–3 покупок в зоне, уровень «{title}»:")
    display(cs.assign(k=cs[f"sharp|{title}"].clip(upper=3)).pivot_table(index="k", columns="n_bin", values="target", aggfunc="mean", observed=True))
    display(cs.assign(k=cs[f"sharp|{title}"].clip(upper=3)).pivot_table(index="k", columns="n_bin", values="target", aggfunc="size", observed=True))

# Профиль на уровне ряда у частых покупателей: есть ли зона вообще
freq = cs.loc[cs["n_days"] >= 6, ["customer_id", "target"]]
t_bins = [-800, -60, -30, 0, 15, 25, 40, 55, 70, 85, 92, 105, 130, 180, 800]
prof, prof_n = {}, {}
for title in ["категория", "пол x категория", "подкатегория", "бренд", "товар"]:
    for n_s, tag in [(2, "2 покупки"), (3, "3 покупки")]:
        st = series[title]
        st = st[st["n"].eq(n_s)].merge(freq, on="customer_id")
        g = st.groupby(pd.cut(st["t"], t_bins), observed=False)["target"]
        prof[f"{title}, {tag}"], prof_n[f"{title}, {tag}"] = g.mean(), g.size()
print("Клиенты с 6+ днями заказов: доля target = 1 по ожидаемому дню следующей покупки в ряду (одна строка — один ряд)")
display(pd.DataFrame(prof))
print("Число рядов:")
display(pd.DataFrame(prof_n))
print("Доля target = 1 у клиентов с 6+ днями заказов в целом:", round(float(freq["target"].mean()), 4))